<a href="https://colab.research.google.com/github/Bibek-Dhakal/ml-playground-for-applied-search-intelligence/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Scoring / Ranking (approached via binary classification).

**Why:** While predicting the exact numerical CTR would technically be a regression task, regression models are unaware of the product's decision thresholds. The actual business decision is sorting a prioritized queue for human SEO reviewers. Therefore, framing this as a binary classification task (e.g., `is_underperforming_ctr`) that outputs a probability score allows us to rank candidates from highest risk to lowest, optimizing the reviewers' limited time.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy Target:** `is_ctr_anomaly` (Binary: 1 or 0)

**Where it comes from:** The ideal outcome is identifying pages where improving the title or meta description is likely to increase CTR. Since I do not have a direct causal label showing whether a metadata change caused a CTR improvement, I will use a **proxy label** based on observed performance. A page will be labeled as a positive example (`1`) if its observed CTR is significantly below the historical average or median for its corresponding `position_tier` and `content_type`, while also meeting a minimum impressions threshold to reduce noise from low-traffic pages.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success Metric:** Precision@50 (Precision@K).

**What "good" means:** The goal is to help human SEO reviewers focus on the highest-value pages first. Rather than maximizing overall accuracy, the model should rank the most actionable CTR opportunities at the top of the review queue. A successful model is one where a high proportion of the top 50 recommended pages are genuine CTR anomalies that are worth investigating (for example, achieving Precision@50 greater than 0.65). This aligns the evaluation metric with the real business decision instead of optimizing for a generic machine learning score.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [3]:
import pandas as pd
import os

# Load the starter dataset (works locally or in Colab)
file_path = "../../data/raw/content_refresh_anonymized.csv"

if not os.path.exists(file_path):
    # Fallback for a fresh Colab session
    file_path = "https://raw.githubusercontent.com/Bibek-Dhakal/ml-playground-for-applied-search-intelligence/refs/heads/main/data/raw/content_refresh_anonymized.csv"


try:
  # Read the dataset
  df = pd.read_csv(file_path)
except FileNotFoundError:
  print(file_path)
  print("File not found. Please check the file path.")
  raise  # Re-raise the exception to stop further execution

# Filter out very low-impression pages to reduce noise
df_lane4 = df[df["impressions_90d"] >= 100].copy()

print("=== Unit of Analysis ===")
print("One row represents one unique pseudonymized content page for a specific client.")
print(f"Total rows in analytical dataset: {len(df_lane4):,}")

print("\nSelected columns:")
display(
    df_lane4[
        [
            "content_id",
            "client_id",
            "position_tier",
            "content_type",
            "impressions_90d",
            "clicks_90d",
            "ctr",
        ]
    ].head()
)

=== Unit of Analysis ===
One row represents one unique pseudonymized content page for a specific client.
Total rows in analytical dataset: 22,006

Selected columns:


,content_id,client_id,position_tier,content_type,impressions_90d,clicks_90d,ctr
0,content_304f48230142,client_f369cb89fc,striking,keyword article,3803,29,0.76
1,content_a1fb4e703a9e,client_4e07408562,page_3_5,keyword article,15320,7,0.05
2,content_9aa793d4d895,client_7f2253d7e2,page_3_5,keyword article,12581,11,0.09
3,content_331d6c4de07b,client_19581e27de,page_1,keyword article,11751,58,0.49
4,content_d99b7a2d90ca,client_3fdba35f04,page_3_5,keyword article,19140,24,0.13


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML wins:** A simple rule such as `IF position < 5 AND ctr < 0.10 THEN review` cannot capture the complexity of real-world search performance. CTR changes non-linearly across search positions and is strongly influenced by factors such as `content_type`, `competition_level`, search intent, and page characteristics. Creating and maintaining hundreds of nested business rules to account for these interactions would be difficult and unreliable. A machine learning model can automatically learn these complex, non-linear relationships and produce a ranked list of pages that better supports human decision-making.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.